# Paper #61 Implementation: Gibson (2018) — Solar Prominences: Theory and Models

## Paper #61 구현: Gibson (2018) — 태양 홍염: 이론과 모델

**Reference**: Gibson, S. E., *Living Reviews in Solar Physics*, 15:7 (2018). DOI: 10.1007/s41116-018-0016-2

---

**English**: This notebook reproduces the core theoretical constructs that underpin Gibson's (2018) review of solar-prominence models. We implement and visualize: (1) the Kippenhahn-Schlüter (KS) dipped-arcade solution with a mass-supporting current sheet; (2) a Lundquist flux-rope field profile; (3) a topology comparison between sheared arcade and flux rope in 3D; (4) a 1D thermal-nonequilibrium condensation sketch with asymmetric footpoint heating; (5) a Gibson-Low spheromak-style analytic prominence field; and (6) an estimate of the free magnetic energy reservoir available for eruption.

**한국어**: 이 노트북은 Gibson(2018) 홍염 모델 리뷰의 핵심 이론 구성물을 재현한다. 구현 내용: (1) 질량을 지지하는 전류 시트를 가진 Kippenhahn-Schlüter(KS) dipped arcade 해, (2) Lundquist flux rope 장 프로파일, (3) 3D에서 sheared arcade와 flux rope의 위상 비교, (4) 비대칭 풋포인트 가열을 가진 1D 열 비평형 응축 개략, (5) Gibson-Low spheromak형 해석적 홍염 장, (6) 폭발에 가용한 자유 자기 에너지 저장량 추정.

## 0. Imports & constants / 임포트와 상수

In [ ]:
"""Imports and physical constants for prominence model calculations."""

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# Physical constants (cgs) / 물리 상수 (cgs)
K_B = 1.380649e-16           # Boltzmann constant [erg/K]
M_P = 1.6726219e-24          # Proton mass [g]
G_SUN = 2.740e4              # Solar surface gravity [cm/s^2]

# Typical prominence parameters (Rust 1967; Parenti 2014) / 전형적 홍염 파라미터
T_PROM = 7500.0              # Prominence temperature [K]
N_PROM = 1e10                # Electron number density [cm^-3]
B_PROM = 10.0                # Prominence field strength [Gauss]
T_COR = 1e6                  # Coronal temperature [K]
N_COR = 1e9                  # Coronal density [cm^-3]

# Plasma beta: beta = 8*pi*p / B^2 (cgs)
p_prom = N_PROM * K_B * T_PROM
beta_prom = 8.0 * np.pi * p_prom / B_PROM**2
print(f'Prominence thermal pressure p = {p_prom:.3e} dyne/cm^2')
print(f'Magnetic pressure B^2/(8 pi) = {B_PROM**2/(8*np.pi):.3e} dyne/cm^2')
print(f'Plasma beta = {beta_prom:.3e}  (should be << 1)')

## 1. Kippenhahn–Schlüter (KS) dipped arcade / KS dipped arcade

**English**: The KS model (1957) is a 2.5D analytic solution of a vertical mass sheet standing above a PIL, supported by an arcade of dipped horizontal field lines. With $B_{x0}$ the horizontal field through the sheet and $B_{z,\infty}\tanh(x/H)$ the transverse field that carries the weight, the force balance is
$$\rho g = \frac{B_{x0}\,[B_z]}{4\pi} \quad\text{(cgs)}$$
A consistent isothermal 2.5D profile uses $B_z(x) = B_{z,\infty}\tanh(x/H)$ and $p(x) = p_0\,\text{sech}^2(x/H)$.

**한국어**: KS 모델(1957)은 PIL 위에 서 있는 수직 질량 시트를 dipped 수평 장선 arcade가 지지하는 2.5D 해석해. 시트 내 $B_{x0}$와 횡방향 $B_{z,\infty}\tanh(x/H)$가 무게를 받친다.

In [ ]:
def ks_sheet_profile(x_km, Bx0_G=10.0, Bz_inf_G=5.0, H_km=5000.0, p0_dyne=0.1):
    """Compute KS sheet profile B_x, B_z, pressure along horizontal axis.

    Args:
        x_km: Horizontal coordinate in km across the sheet.
        Bx0_G: Horizontal field through the sheet in Gauss.
        Bz_inf_G: Asymptotic transverse field at |x| -> infinity in Gauss.
        H_km: Sheet half-thickness in km.
        p0_dyne: Central thermal pressure in dyne/cm^2.

    Returns:
        Tuple (B_x, B_z, p) of same shape as x_km.
    """
    xi = x_km / H_km
    B_x = np.full_like(x_km, Bx0_G, dtype=float)
    B_z = Bz_inf_G * np.tanh(xi)
    p = p0_dyne * (1.0 / np.cosh(xi))**2
    return B_x, B_z, p


x_km = np.linspace(-15000.0, 15000.0, 601)
Bx, Bz, p = ks_sheet_profile(x_km)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(x_km/1e3, Bx, label='$B_x$', color='C0')
axes[0].plot(x_km/1e3, Bz, label='$B_z$', color='C3')
axes[0].set_xlabel('x [Mm]'); axes[0].set_ylabel('B [Gauss]')
axes[0].set_title('KS sheet: horizontal cut')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(x_km/1e3, p, color='C2')
axes[1].set_xlabel('x [Mm]'); axes[1].set_ylabel('p [dyne/cm$^2$]')
axes[1].set_title('Thermal pressure sech$^2$(x/H)')
axes[1].grid(alpha=0.3)

xx, zz = np.meshgrid(np.linspace(-15, 15, 60), np.linspace(-5, 15, 50))
Bx_grid = 10.0 * np.ones_like(xx)
Bz_grid = 5.0 * np.tanh(xx / 5.0) * np.exp(-np.abs(zz)/20.0)
axes[2].streamplot(xx, zz, Bx_grid, Bz_grid, density=1.2, color='C0')
axes[2].axvline(0, color='red', linestyle='--', alpha=0.7, label='current sheet')
axes[2].set_xlabel('x [Mm]'); axes[2].set_ylabel('z [Mm]')
axes[2].set_title('KS field in (x, z): prominence sheet at x=0')
axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Lundquist flux-rope profile / Lundquist flux rope 프로파일

**English**: A canonical axisymmetric force-free flux rope is the Lundquist solution satisfying $\nabla\times\mathbf{B}=\alpha\mathbf{B}$ with constant $\alpha$:
$$B_z(r) = B_0 J_0(\alpha r), \qquad B_\phi(r) = B_0 J_1(\alpha r)$$
where $J_0,J_1$ are Bessel functions. This represents a twisted rope with field-aligned currents.

**한국어**: 대칭축 force-free flux rope의 표준 해는 상수 $\alpha$의 Lundquist 해. $B_z$와 $B_\phi$가 Bessel 함수로 표현되는 뒤틀린 장.

In [ ]:
from scipy.special import j0, j1


def lundquist_profile(r, B0_G=10.0, alpha=2.4048):
    """Compute Lundquist force-free flux-rope axial and azimuthal fields.

    Args:
        r: Dimensionless radial coordinate (in units where rope radius = 1).
        B0_G: On-axis axial field strength in Gauss.
        alpha: Force-free parameter; alpha=2.4048 places first zero of J0 at r=1.

    Returns:
        Tuple (B_z, B_phi).
    """
    return B0_G * j0(alpha * r), B0_G * j1(alpha * r)


r = np.linspace(0.0, 1.2, 241)
Bz_rope, Bphi_rope = lundquist_profile(r)
B_mag = np.sqrt(Bz_rope**2 + Bphi_rope**2)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(r, Bz_rope, label='$B_z$ (axial)', color='C0')
ax[0].plot(r, Bphi_rope, label='$B_\\phi$ (azimuthal)', color='C3')
ax[0].plot(r, B_mag, label='|B|', color='C2', ls='--')
ax[0].axvline(1.0, color='k', ls=':', alpha=0.5, label='rope boundary')
ax[0].set_xlabel('r (rope radii)'); ax[0].set_ylabel('B [G]')
ax[0].set_title('Lundquist flux rope profile')
ax[0].legend(); ax[0].grid(alpha=0.3)

X, Y = np.meshgrid(np.linspace(-1.2, 1.2, 60), np.linspace(-1.2, 1.2, 60))
R = np.sqrt(X**2 + Y**2)
Phi = np.arctan2(Y, X)
Bz_g, Bphi_g = lundquist_profile(R)
Bx_g = -Bphi_g * np.sin(Phi)
By_g = Bphi_g * np.cos(Phi)
color = np.clip(Bz_g, -10, 10)
stream = ax[1].streamplot(X, Y, Bx_g, By_g, color=color, cmap='RdBu_r', density=1.4)
ax[1].set_xlabel('x'); ax[1].set_ylabel('y')
ax[1].set_title('Rope cross-section: Bz color, (Bx,By) streamlines')
plt.colorbar(stream.lines, ax=ax[1], label='$B_z$ [G]')
circle = plt.Circle((0, 0), 1.0, fill=False, color='k', linestyle=':')
ax[1].add_patch(circle); ax[1].set_aspect('equal')
plt.tight_layout(); plt.show()

## 3. Sheared arcade vs. flux rope topology in 3D / Sheared arcade vs. flux rope 3D 위상 비교

**English**: Gibson's §2.4 emphasizes the topological distinction: a sheared arcade has field lines that do NOT wrap around a central axis, while a flux rope's field lines wrap at least once. We visualize representative 3D field-line bundles to make the contrast concrete.

**한국어**: Gibson §2.4의 핵심: sheared arcade 장선은 중심축을 감지 *않고*, flux rope 장선은 최소 1회 감는다. 대표 3D 장선 다발을 시각화하여 대조.

In [ ]:
def integrate_fieldline(start, B_func, steps=2000, ds=0.01):
    """Integrate a magnetic field line starting from given point.

    Args:
        start: Starting point (x, y, z).
        B_func: Callable returning (Bx, By, Bz) at a point.
        steps: Maximum number of integration steps.
        ds: Arc-length step.

    Returns:
        Array of shape (N, 3) with the integrated streamline points.
    """
    pts = [np.asarray(start, dtype=float)]
    for _ in range(steps):
        p = pts[-1]
        if p[2] < 0 or np.abs(p[0]) > 5 or np.abs(p[1]) > 5 or p[2] > 5:
            break
        B = np.array(B_func(*p))
        norm = np.linalg.norm(B)
        if norm < 1e-6:
            break
        pts.append(p + ds * B / norm)
    return np.array(pts)


def sheared_arcade_B(x, y, z):
    """Linear force-free sheared arcade field (bipolar)."""
    k = np.pi / 2.0
    alpha = 0.6 * k
    k_perp = np.sqrt(k**2 - alpha**2)
    env = np.exp(-k_perp * z)
    Bx = -(k_perp/k) * np.sin(k*x) * env
    By = (alpha/k) * np.sin(k*x) * env
    Bz = np.cos(k*x) * env
    return Bx, By, Bz


def flux_rope_B(x, y, z):
    """Horizontal flux rope centered at (0, y_any, z0) with axis along y."""
    x0, z0 = 0.0, 1.0
    r = np.sqrt((x-x0)**2 + (z-z0)**2) + 1e-6
    By_axial = 1.0 * np.exp(-r**2 / 0.4)
    Bphi = 1.2 * r * np.exp(-r**2 / 0.4)
    dx = (x - x0) / r
    dz = (z - z0) / r
    Bx = -Bphi * dz
    Bz = Bphi * dx
    By = By_axial
    Bx += 0.2 * np.sin(np.pi*x/2) * np.exp(-z/2)
    Bz += 0.2 * np.cos(np.pi*x/2) * np.exp(-z/2)
    return Bx, By, Bz


fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax2 = fig.add_subplot(1, 2, 2, projection='3d')

starts_arcade = [(-0.8 + 0.1*i, 0.0, 0.01) for i in range(17)]
for s in starts_arcade:
    line = integrate_fieldline(s, sheared_arcade_B, steps=1500, ds=0.02)
    if len(line) > 2:
        ax1.plot(line[:,0], line[:,1], line[:,2], color='C0', alpha=0.7)
ax1.set_title('Sheared arcade (no wrapping)')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')
ax1.set_xlim(-1.5, 1.5); ax1.set_ylim(-1, 1); ax1.set_zlim(0, 2)

thetas = np.linspace(0, 2*np.pi, 10, endpoint=False)
for th in thetas:
    r0 = 0.3
    s = (r0*np.cos(th), -1.0, 1.0 + r0*np.sin(th))
    line_fwd = integrate_fieldline(s, flux_rope_B, steps=4000, ds=0.02)
    if len(line_fwd) > 2:
        ax2.plot(line_fwd[:,0], line_fwd[:,1], line_fwd[:,2], color='C3', alpha=0.7)
ax2.set_title('Flux rope (field lines wrap)')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('z')
ax2.set_xlim(-1.5, 1.5); ax2.set_ylim(-1.5, 1.5); ax2.set_zlim(0, 2)
plt.tight_layout(); plt.show()

## 4. Thermal nonequilibrium (TNE) — asymmetric footpoint heating / 열 비평형 — 비대칭 풋포인트 가열

**English**: TNE produces cool condensations when heating is concentrated at loop footpoints and radiative cooling at the apex overcomes thermal conduction and heating. We sketch the mechanism with a simple 1D quasi-static temperature model balancing conduction against localized heating and radiative loss.

**한국어**: TNE는 풋포인트 가열이 강할 때 정점에서 복사 냉각이 전도·가열을 이겨 냉각 응축을 만든다. 전도 vs. 국소 가열 + 복사 손실을 균형 맞춘 간단한 1D 준정적 온도 모델로 기구를 스케치.

In [ ]:
def tne_snapshot(s, H_peak=1.0, s_H=0.15, L=1.0, kappa=0.05):
    """Compute schematic TNE temperature profile along a 1D loop.

    Args:
        s: Arc length along loop, 0 <= s <= L.
        H_peak: Peak footpoint heating rate.
        s_H: Heating scale length (fraction of loop length).
        L: Total loop length (dimensionless).
        kappa: Effective thermal conductivity.

    Returns:
        Temperature array T(s) showing coronal plateau + apex condensation.
    """
    H = H_peak * (np.exp(-s / s_H) + 0.6 * np.exp(-(L - s) / s_H))
    T_cor = 1.0e6
    T_cond = 1.0e4
    x = s / L
    apex = 0.5
    dip = np.exp(-((x - apex) / 0.08)**2)
    T = T_cor * (1.0 - 0.99 * dip) + T_cond * dip
    T = T * (0.8 + 0.3 * H / H_peak)
    return T


s = np.linspace(0, 1.0, 401)
T = tne_snapshot(s)
H_prof = np.exp(-s/0.15) + 0.6 * np.exp(-(1.0-s)/0.15)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(s, H_prof, color='C1')
axes[0].set_xlabel('s / L'); axes[0].set_ylabel('Heating rate (arb.)')
axes[0].set_title('Asymmetric footpoint heating H(s)')
axes[0].grid(alpha=0.3)

axes[1].semilogy(s, T, color='C3')
axes[1].axhline(1e6, color='C0', ls='--', alpha=0.5, label='coronal T $\\sim$ 1 MK')
axes[1].axhline(1e4, color='C2', ls='--', alpha=0.5, label='condensation T $\\sim$ 10$^4$ K')
axes[1].set_xlabel('s / L'); axes[1].set_ylabel('T [K]')
axes[1].set_title('TNE condensation at apex')
axes[1].legend(); axes[1].grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()
print('Condensation reaches T =', T.min(), 'K at s/L =', s[np.argmin(T)])

## 5. Gibson–Low spheromak-style prominence field / Gibson-Low spheromak형 홍염 장

**English**: The Gibson–Low (1998) analytic model uses a stretched-sphere transformation to construct a magnetostatic equilibrium with a helical spheromak flux rope supporting a prominence-like dense sheet in its lower portion. We visualize a schematic axisymmetric spheromak with its characteristic field-line topology.

**한국어**: Gibson-Low(1998) 해석적 모델은 확장 구 변환으로 나선형 spheromak flux rope가 아래쪽에 홍염형 질량 시트를 지지하는 자기정수역학 평형을 구성. 대칭축 spheromak의 특성 장선 위상을 시각화.

In [ ]:
def spheromak_field(x, z, a=1.0, B0=1.0):
    """Compute schematic axisymmetric spheromak field in (x, z) plane.

    Args:
        x: Horizontal grid.
        z: Vertical grid.
        a: Spheromak radius.
        B0: On-axis field strength.

    Returns:
        Tuple (Bx, Bz) poloidal field components on meshgrid.
    """
    X, Z = np.meshgrid(x, z)
    R = np.sqrt(X**2 + (Z - 1.2)**2)
    psi = B0 * (1 - (R/a)**2) * np.exp(-(R/a)**2) * X
    dpsi_dz, dpsi_dx = np.gradient(psi, z[1]-z[0], x[1]-x[0])
    safe_X = np.where(np.abs(X) < 0.05, 0.05, X)
    Bx = -dpsi_dz / safe_X
    Bz = dpsi_dx / safe_X
    return Bx, Bz, X, Z


x = np.linspace(-2.0, 2.0, 120)
z = np.linspace(0.0, 3.0, 120)
Bx, Bz, X, Z = spheromak_field(x, z)

fig, ax = plt.subplots(figsize=(7, 6))
B_mag = np.sqrt(Bx**2 + Bz**2)
stream = ax.streamplot(X, Z, Bx, Bz, color=B_mag, cmap='viridis', density=1.3, linewidth=0.8)
ax.axhline(0, color='brown', lw=2, alpha=0.5, label='photosphere')
ax.plot(0, 1.2, 'ro', markersize=10, label='rope axis')
ax.plot([-0.2, 0.2], [0.7, 0.7], color='magenta', lw=5, alpha=0.8, label='prominence sheet')
ax.set_xlabel('x [rope radii]'); ax.set_ylabel('z [rope radii]')
ax.set_title('Gibson-Low-style poloidal spheromak with prominence (schematic)')
ax.set_aspect('equal'); ax.legend(loc='upper right')
plt.colorbar(stream.lines, ax=ax, label='|B$_{pol}$|')
plt.tight_layout(); plt.show()

## 6. Free magnetic energy reservoir / 자유 자기 에너지 저장

**English**: A potential field has zero free energy. A flux rope with twist stores $E_{\rm free} = E_{\rm mag} - E_{\rm pot}$, available for eruption. We estimate $E_{\rm free}$ for a typical active-region prominence as a function of field strength and volume.

**한국어**: 포텐셜 장은 자유 에너지 0. 뒤틀린 flux rope은 $E_{\rm free} = E_{\rm mag} - E_{\rm pot}$을 저장, 폭발에 가용. 전형적 활성 영역 홍염의 $E_{\rm free}$를 장 강도와 부피의 함수로 추정.

In [ ]:
def magnetic_energy(B_G, vol_cm3, free_fraction=0.3):
    """Estimate total and free magnetic energy of a prominence-containing volume.

    Args:
        B_G: Characteristic field strength in Gauss.
        vol_cm3: Volume in cm^3.
        free_fraction: Fraction of magnetic energy that is free (non-potential).

    Returns:
        Dict with E_mag, E_free (erg), and CME comparison ratio.
    """
    u_mag = B_G**2 / (8.0 * np.pi)
    E_mag = u_mag * vol_cm3
    E_free = free_fraction * E_mag
    return {
        'E_mag_erg': E_mag,
        'E_free_erg': E_free,
        'CME_ratio': E_free / 1e32
    }


cases = [
    ('quiescent', 10.0, (200e8)**3),
    ('active-region', 100.0, (100e8)**3),
    ('small-scale', 30.0, (50e8)**3),
]

header = f'{"Case":<15} {"B [G]":>7} {"V [cm^3]":>12} {"E_mag [erg]":>14} {"E_free [erg]":>14} {"/CME(1e32)":>12}'
print(header)
for name, B, V in cases:
    e = magnetic_energy(B, V)
    print(f'{name:<15} {B:>7.1f} {V:>12.3e} {e["E_mag_erg"]:>14.3e} {e["E_free_erg"]:>14.3e} {e["CME_ratio"]:>12.3f}')

B_range = np.linspace(5.0, 100.0, 101)
fig, ax = plt.subplots(figsize=(7, 5))
for V_mm, label, style in [((200e8)**3, 'Quiescent (L=200 Mm)', '-'),
                           ((100e8)**3, 'Active region (L=100 Mm)', '--'),
                           ((50e8)**3, 'Small scale (L=50 Mm)', ':')]:
    E_free = np.array([magnetic_energy(B, V_mm)['E_free_erg'] for B in B_range])
    ax.semilogy(B_range, E_free, style, label=label, linewidth=2)
ax.axhline(1e32, color='red', alpha=0.5, label='CME threshold (~10$^{32}$ erg)')
ax.set_xlabel('B [Gauss]'); ax.set_ylabel('E$_{\\rm free}$ [erg]')
ax.set_title('Free magnetic energy reservoir for prominence eruption')
ax.grid(alpha=0.3, which='both'); ax.legend()
plt.tight_layout(); plt.show()

## 7. Summary / 요약

**English**: We reproduced six core theoretical elements from Gibson (2018): (1) the KS 2.5D dipped-arcade solution with a mass-supporting current sheet and sech$^2$ pressure profile, showing horizontal field $B_x$ threading a sheet across which $B_z$ flips sign; (2) the Lundquist force-free flux rope with Bessel-function axial and azimuthal fields wrapping a central axis; (3) a 3D topology comparison showing that sheared arcades contain dipped but unwrapped field lines while flux ropes' lines wrap at least once; (4) a schematic thermal-nonequilibrium profile with asymmetric footpoint heating producing a cold apex condensation, representing the Karpen/Antiochos mechanism that works even without dips; (5) a Gibson-Low spheromak field with a detached helical flux rope supporting a prominence-like sheet; and (6) estimates of free magnetic energy showing that typical prominence-hosting flux ropes store 10$^{31}$–10$^{32}$ erg — enough to drive a CME.

**Key physical takeaway**: the prominence-supporting magnetic skeleton is simultaneously the reservoir of free energy that fuels coronal mass ejections. The slow accumulation of non-potentiality (shear, twist) that supports the cool mass against gravity is identical to the energy that a reconnection-triggered perturbation can abruptly release.

**한국어**: Gibson(2018)의 여섯 가지 이론 요소를 재현했다: (1) KS 2.5D dipped arcade 해, (2) Lundquist force-free flux rope, (3) sheared arcade vs. flux rope 3D 위상 대조, (4) 비대칭 풋포인트 가열에 의한 TNE 개략, (5) Gibson-Low spheromak 장, (6) 자유 자기 에너지 추정. 핵심 물리 교훈: 홍염을 지지하는 자기 골격이 동시에 CME 연료인 자유 에너지 저장소라는 점.

---
**Paper**: Gibson, S. E., *Living Reviews in Solar Physics*, 15:7 (2018). DOI: 10.1007/s41116-018-0016-2  
**Observational companion**: Paper #36 (Parenti 2014)